# LoRA Lesson

This notebook walks through a small reinforcement fine-tuning example using GRPO and LoRA on a simple arithmetic task.

## Imports

Import the standard library, PyTorch, dataset tools, Transformers components, and the TRL and PEFT classes used throughout the lesson.

In [1]:
from typing import Any
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOTrainer, GRPOConfig
from peft import LoraConfig, TaskType
from rewards import compute_reward, correctness_reward, format_reward, think_reward
from mlflow_tracking import (
    initialize_tracking,
    log_eval_result,
    log_history,
    log_json_artifact,
    log_metrics,
    log_params,
    start_child_run,
    start_parent_run,
)


/Users/jim/Desktop/genai/rft-learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
import shutil

for directory in ["grpo-arithmetic-lora-demo", "grpo-arithmetic-lora-adapter"]:
    shutil.rmtree(Path(directory), ignore_errors=True)

print("Deleted any existing GRPO output directories.")


Deleted any existing GRPO output directories.


## Constants

Define the dataset bounds and the pretrained instruction model that will be evaluated and then fine-tuned.

In [3]:
MAX_A = 21
MAX_B = 11

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
# model_name = "Qwen/Qwen2.5-1.5B-Instruct"


## Dataset Builder

Create a helper function that generates arithmetic prompts and the expected answers for a small synthetic training set.

In [4]:
def make_dataset() -> Dataset:
    """Build a small arithmetic dataset with strict output-format instructions.

    Args:
        None.

    Returns:
        Dataset: A Hugging Face dataset containing prompt and answer pairs.
    """
    rows = []

    for a in range(1, MAX_A):
        for b in range(1, MAX_B):
            prompt_text = f"What is {a} + {b}? Respond exactly as <think>{a} + {b}</think><answer>...</answer>"
            rows.append({
                "prompt": [{"role": "user", "content": prompt_text}],
                "answer": str(a + b),
            })

    return Dataset.from_list(rows)


def readable_prompt(prompt: Any) -> str:
    """Extract readable text from a conversational or legacy string prompt."""
    if isinstance(prompt, str):
        return prompt

    return "\n".join(
        message.get("content", "")
        for message in prompt
        if isinstance(message, dict) and isinstance(message.get("content"), str)
    )


## Dataset Split

Build the dataset and split it into training and test subsets so we can compare behavior before and after fine-tuning.

In [5]:
dataset = make_dataset()
split = dataset.train_test_split(test_size=0.25, seed=42)

train_dataset = split["train"]
test_dataset = split["test"]

# Keep the first 5 training prompts so we can filter GRPO completion logs later.
num_first_train_records = min(5, len(train_dataset))
first_five_train_prompts = {
    readable_prompt(prompt)
    for prompt in train_dataset.select(range(num_first_train_records))["prompt"]
}

# show the first examples from the training dataset
print(f"First {num_first_train_records} examples from the training dataset:")
for i in range(num_first_train_records):
    row = train_dataset[i]
    print({"prompt": readable_prompt(row["prompt"]), "answer": row["answer"]})

# show up to the first 5 examples from the test dataset
num_test_examples_to_show = min(5, len(test_dataset))
print(f"First {num_test_examples_to_show} examples from the test dataset:")
for i in range(num_test_examples_to_show):
    row = test_dataset[i]
    print({"prompt": readable_prompt(row["prompt"]), "answer": row["answer"]})

First 5 examples from the training dataset:
{'prompt': 'What is 9 + 3? Respond exactly as <think>9 + 3</think><answer>...</answer>', 'answer': '12'}
{'prompt': 'What is 10 + 9? Respond exactly as <think>10 + 9</think><answer>...</answer>', 'answer': '19'}
{'prompt': 'What is 9 + 10? Respond exactly as <think>9 + 10</think><answer>...</answer>', 'answer': '19'}
{'prompt': 'What is 3 + 8? Respond exactly as <think>3 + 8</think><answer>...</answer>', 'answer': '11'}
{'prompt': 'What is 11 + 9? Respond exactly as <think>11 + 9</think><answer>...</answer>', 'answer': '20'}
First 5 examples from the test dataset:
{'prompt': 'What is 16 + 3? Respond exactly as <think>16 + 3</think><answer>...</answer>', 'answer': '19'}
{'prompt': 'What is 3 + 7? Respond exactly as <think>3 + 7</think><answer>...</answer>', 'answer': '10'}
{'prompt': 'What is 12 + 6? Respond exactly as <think>12 + 6</think><answer>...</answer>', 'answer': '18'}
{'prompt': 'What is 5 + 7? Respond exactly as <think>5 + 7</think>

## Shared Reward Module

Use the reward computations imported from `rewards.py` for evaluation and GRPO training.

## Response Generation Helper

Define a helper that formats a prompt as a chat conversation, runs generation, and decodes only the new tokens.

In [6]:
def generate_response(model: Any, tokenizer: Any, prompt: Any) -> str:
    """Generate a deterministic response for a conversational prompt.

    Args:
        model: The causal language model used for generation.
        tokenizer: The tokenizer used to format and decode the prompt.
        prompt: A conversational prompt or legacy prompt string.

    Returns:
        str: The decoded generated response text.
    """
    messages = [{"role": "user", "content": prompt}] if isinstance(prompt, str) else prompt
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            bos_token_id=tokenizer.bos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


## Evaluation Helper

Define the evaluation routine that generates predictions across the test set and prints accuracy, format compliance, think compliance, reward, and sample outputs.

In [7]:
def evaluate_model(model: Any, tokenizer: Any, eval_dataset: Dataset, label: str) -> dict[str, Any]:
    """Run evaluation on a dataset, print summary metrics, and return structured results."""
    model.eval()

    total = len(eval_dataset)
    correct = 0
    formatted = 0
    think_valid = 0
    total_reward = 0.0
    total_format_reward = 0.0
    total_correctness_reward = 0.0
    total_think_reward = 0.0

    samples = []
    rows = []

    for row in eval_dataset:
        prompt = row["prompt"]
        expected = row["answer"]

        text = generate_response(model, tokenizer, prompt)
        reward_result = compute_reward(text, expected, prompt)
        reward = reward_result.total_reward

        formatted += int(reward_result.is_formatted)
        correct += int(reward_result.is_correct)
        think_valid += int(reward_result.is_think_valid)
        total_reward += reward
        total_format_reward += reward_result.format_reward
        total_correctness_reward += reward_result.correctness_reward
        total_think_reward += reward_result.think_reward

        eval_row = {
            "prompt": readable_prompt(prompt),
            "expected": expected,
            "generated": text,
            "predicted": reward_result.predicted_answer,
            "is_correct": reward_result.is_correct,
            "is_formatted": reward_result.is_formatted,
            "is_think_valid": reward_result.is_think_valid,
            "format_reward": reward_result.format_reward,
            "correctness_reward": reward_result.correctness_reward,
            "think_reward": reward_result.think_reward,
            "reward": reward,
        }
        rows.append(eval_row)

        if len(samples) < 5:
            samples.append({
                "prompt": readable_prompt(prompt),
                "expected": expected,
                "generated": text,
                "predicted": reward_result.predicted_answer,
                "think_reward": reward_result.think_reward,
                "reward": reward,
            })

    metrics = {
        "prompt_count": total,
        "completion_count": total,
        "accuracy": correct / total,
        "format_compliance": formatted / total,
        "think_compliance": think_valid / total,
        "avg_reward": total_reward / total,
        "avg_format_reward": total_format_reward / total,
        "avg_correctness_reward": total_correctness_reward / total,
        "avg_think_reward": total_think_reward / total,
    }
    result = {
        "label": label,
        "metrics": metrics,
        "settings": {
            "num_generations": 1,
            "do_sample": False,
        },
        "samples": samples,
        "rows": rows,
    }

    print(f"\n=== {label} ===")
    print(f"Answer accuracy:   {correct}/{total} = {metrics['accuracy']:.2%}")
    print(f"Format compliance: {formatted}/{total} = {metrics['format_compliance']:.2%}")
    print(f"Think compliance:  {think_valid}/{total} = {metrics['think_compliance']:.2%}")
    print(f"Average reward:    {metrics['avg_reward']:.3f}")

    print("\nSample generations:")
    for ex in samples:
        print("-" * 60)
        print("Prompt:   ", ex["prompt"])
        print("Expected: ", ex["expected"])
        print("Generated:", ex["generated"])
        print("Predicted:", ex["predicted"])
        print("Think reward:", ex["think_reward"])
        print("Reward:   ", ex["reward"])

    return result


## Tokenizer Setup

Load the tokenizer for the base instruction model so prompts can be formatted and outputs decoded.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_name)


## Base Model Setup

Load the pretrained causal language model and choose a practical dtype depending on whether CUDA is available.

In [9]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:01<00:00, 261.63it/s]


## Baseline Evaluation

Measure how the base model performs on the held-out arithmetic examples before applying GRPO and LoRA.

In [10]:
tracking_setup = initialize_tracking(experiment_name="rft-learning")
parent_run_manager = start_parent_run(
    notebook_name="rft-lora-lesson.ipynb",
    notebook_type="rft",
    tags={"model_name": model_name},
)
parent_run = parent_run_manager.__enter__()

log_params(
    {
        "model_name": model_name,
        "notebook_type": "rft",
        "dataset": {
            "train_size": len(train_dataset),
            "test_size": len(test_dataset),
            "split_seed": 42,
        },
    },
    prefix="run",
)
log_json_artifact(
    "rft_run_context.json",
    {
        "tracking": {
            "tracking_uri": tracking_setup.tracking_uri,
            "experiment_name": tracking_setup.experiment_name,
            "tracking_path": str(tracking_setup.tracking_path),
        },
        "dataset": {
            "train_size": len(train_dataset),
            "test_size": len(test_dataset),
            "split_seed": 42,
        },
    },
    artifact_path="run_context",
)

baseline_eval_result = evaluate_model(
    base_model,
    tokenizer,
    test_dataset,
    label="Before GRPO + LoRA"
)

with start_child_run("baseline_eval"):
    log_eval_result("baseline_eval", baseline_eval_result)

log_metrics(baseline_eval_result["metrics"], prefix="baseline")



=== Before GRPO + LoRA ===
Answer accuracy:   0/50 = 0.00%
Format compliance: 0/50 = 0.00%
Think compliance:  0/50 = 0.00%
Average reward:    0.000

Sample generations:
------------------------------------------------------------
Prompt:    What is 16 + 3? Respond exactly as <think>16 + 3</think><answer>...</answer>
Expected:  19
Generated: The result of 16 + 3 is 19.
Predicted: 
Think reward: 0.0
Reward:    0.0
------------------------------------------------------------
Prompt:    What is 3 + 7? Respond exactly as <think>3 + 7</think><answer>...</answer>
Expected:  10
Generated: 3 + 7 = 10
Predicted: 
Think reward: 0.0
Reward:    0.0
------------------------------------------------------------
Prompt:    What is 12 + 6? Respond exactly as <think>12 + 6</think><answer>...</answer>
Expected:  18
Generated: The result of 12 + 6 is 18.
Predicted: 
Think reward: 0.0
Reward:    0.0
------------------------------------------------------------
Prompt:    What is 5 + 7? Respond exactly as <t

{'baseline.prompt_count': 50.0,
 'baseline.completion_count': 50.0,
 'baseline.accuracy': 0.0,
 'baseline.format_compliance': 0.0,
 'baseline.think_compliance': 0.0,
 'baseline.avg_reward': 0.0,
 'baseline.avg_format_reward': 0.0,
 'baseline.avg_correctness_reward': 0.0,
 'baseline.avg_think_reward': 0.0}

## LoRA Configuration

Configure the LoRA adapter modules and hyperparameters that will be attached during GRPO training.

In [11]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


## GRPO Training Arguments

Set the GRPO hyperparameters, including output location, batch sizes, number of generations, and completion length.

In [12]:
training_args = GRPOConfig(
    output_dir="grpo-arithmetic-lora-demo",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=64,
    num_train_epochs=4,
    logging_steps=10,
    learning_rate=5e-5,
    log_completions=True,
)

## Trainer Construction

Create the GRPO trainer by connecting the base model, reward functions, training dataset, and LoRA configuration.

In [13]:
trainer = GRPOTrainer(
    model=base_model,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=[format_reward, correctness_reward, think_reward],
    peft_config=lora_config,
)

log_params(
    {
        "lora": lora_config,
        "training": training_args,
    },
)
log_json_artifact(
    "rft_training_config.json",
    {
        "base_model_name": model_name,
        "lora_config": lora_config,
        "training_args": training_args,
    },
    artifact_path="configs",
)

sample_prompt = test_dataset[0]["prompt"]
standalone_prompt_ids = tokenizer.apply_chat_template(
    sample_prompt,
    add_generation_prompt=True,
    tokenize=True,
)["input_ids"]
trainer_prompt_ids, _, _ = trainer._tokenize_prompts([sample_prompt])
prompt_tokenization_matches = standalone_prompt_ids == trainer_prompt_ids[0]
print(f"Standalone and TRL prompt token IDs match: {prompt_tokenization_matches}")
assert prompt_tokenization_matches, "Standalone evaluation and GRPO training tokenize prompts differently"


Standalone and TRL prompt token IDs match: True


## Training And Saving

Run GRPO training and save the resulting adapter weights so they can be reused later.

## Plain Log Text (No ANSI Colors)

Disable colorized terminal output so training log text is easier to read in notebook outputs.

In [14]:
import os
from IPython.display import HTML, display
import trl.trainer.grpo_trainer as grpo_trainer_module

# Make TRL/Rich log tables render without color styling.
os.environ["NO_COLOR"] = "1"

# Force notebook output text to black for both ANSI and HTML-rendered tables.
display(HTML("""
<style>
.jp-OutputArea .ansi-yellow-fg,
.jp-OutputArea .ansi-green-fg,
.jp-OutputArea .ansi-blue-fg,
.jp-OutputArea .ansi-magenta-fg,
.jp-OutputArea .ansi-cyan-fg,
.jp-OutputArea .ansi-red-fg,
.jp-OutputArea .ansi-bright-black-fg,
.jp-OutputArea .ansi-bright-red-fg,
.jp-OutputArea .ansi-bright-green-fg,
.jp-OutputArea .ansi-bright-yellow-fg,
.jp-OutputArea .ansi-bright-blue-fg,
.jp-OutputArea .ansi-bright-magenta-fg,
.jp-OutputArea .ansi-bright-cyan-fg,
.jp-OutputArea .ansi-bright-white-fg,
.jp-OutputArea span[style*="color"],
.jp-OutputArea pre[style*="color"],
.jp-OutputArea-output table,
.jp-OutputArea-output table *,
.jp-OutputArea-output pre,
.jp-OutputArea-output code {
    color: #000000 !important;
    text-decoration-color: #000000 !important;
}
</style>
"""))


def _print_prompt_completions_sample_plain(
    prompts,
    completions,
    rewards,
    advantages,
    step,
    num_samples=None,
    extra=None,
):
    """Replacement for TRL rich logger that prints plain text with no color."""
    extra = extra or {}

    rows = []
    for i in range(len(prompts)):
        row = {
            "prompt": str(prompts[i]),
            "completion": str(completions[i]),
            "advantage": f"{advantages[i]:.2f}",
        }
        for reward_name, reward_values in rewards.items():
            row[reward_name] = f"{reward_values[i]:.2f}"
        for extra_name, extra_values in extra.items():
            row[extra_name] = str(extra_values[i])
        rows.append(row)

    if num_samples is not None and 0 < num_samples < len(rows):
        rows = rows[:num_samples]

    print(f"\n=== Step {step} completions (plain text) ===")
    for idx, row in enumerate(rows, start=1):
        print("-" * 80)
        print(f"Row {idx}")
        print(f"Prompt: {row['prompt']}")
        print(f"Completion: {row['completion']}")
        for key, value in row.items():
            if key not in {"prompt", "completion"}:
                print(f"{key}: {value}")


# Monkeypatch TRL GRPO logging to avoid Rich colorized tables.
grpo_trainer_module.print_prompt_completions_sample = _print_prompt_completions_sample_plain

print("Applied plain-text GRPO completion logger (no color). Re-run trainer.train() to use it.")

Applied plain-text GRPO completion logger (no color). Re-run trainer.train() to use it.


In [15]:
with start_child_run("training"):
    train_result = trainer.train()
    adapter_output_dir = "grpo-arithmetic-lora-adapter"
    trainer.save_model(adapter_output_dir)
    training_summary = {
        "output_dir": training_args.output_dir,
        "adapter_output_dir": adapter_output_dir,
        "train_result_metrics": train_result.metrics,
    }
    grpo_logs = [row for row in trainer.state.log_history if "reward" in row]
    log_params({"output_dir": training_args.output_dir, "adapter_output_dir": adapter_output_dir}, prefix="training")
    log_metrics(train_result.metrics)
    log_json_artifact("rft_training_summary.json", training_summary, artifact_path="training")
    log_history(grpo_logs, "grpo_training_history", metric_prefix="grpo")

log_metrics(train_result.metrics, prefix="training")


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/jim/Desktop/genai/rft-learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,-0.008517
20,0.001674
30,-0.000715
40,0.000000
50,0.000483
60,0.000872
70,-0.000000



=== Step 10 completions (plain text) ===
--------------------------------------------------------------------------------
Row 1
Prompt: system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What is 15 + 6? Respond exactly as <think>15 + 6</think><answer>...</answer>
assistant

Completion: <think>15 + 6</think>
<answer>21</answer>
advantage: 0.00
format_reward: 0.50
correctness_reward: 1.00
think_reward: 1.00
--------------------------------------------------------------------------------
Row 2
Prompt: system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What is 15 + 6? Respond exactly as <think>15 + 6</think><answer>...</answer>
assistant

Completion: <think>15 + 6</think>
<answer>21</answer>
advantage: 0.00
format_reward: 0.50
correctness_reward: 1.00
think_reward: 1.00
--------------------------------------------------------------------------------
Row 3
Prompt: system
You are Qwen, created by Alibaba Cloud. You are a helpful as

{'training.train_runtime': 294.0827,
 'training.train_samples_per_second': 2.04,
 'training.train_steps_per_second': 0.245,
 'training.total_flos': 0.0,
 'training.train_loss': -0.0008615380825681819,
 'training.num_tokens': 174428.0,
 'training.completions/mean_length': 19.53125,
 'training.completions/min_length': 18.0,
 'training.completions/max_length': 23.5,
 'training.completions/clipped_ratio': 0.0,
 'training.completions/mean_terminated_length': 19.53125,
 'training.completions/min_terminated_length': 18.0,
 'training.completions/max_terminated_length': 23.5,
 'training.rewards/format_reward/mean': 0.4921875,
 'training.rewards/format_reward/std': 0.04419417306780815,
 'training.rewards/correctness_reward/mean': 1.0,
 'training.rewards/correctness_reward/std': 0.0,
 'training.rewards/think_reward/mean': 0.96875,
 'training.rewards/think_reward/std': 0.12296734005212784,
 'training.reward': 2.4609375,
 'training.reward_std': 0.15695405006408691,
 'training.frac_reward_zero_std':

## GRPO Training Metrics

Summarize the GRPO-native metrics captured during training so the run can be interpreted with reward, KL, entropy, and clipping signals instead of the near-zero policy loss alone.

In [16]:
grpo_logs = [row for row in trainer.state.log_history if "reward" in row]

metric_columns = [
    ("reward", "reward"),
    ("reward_std", "reward_std"),
    ("rewards/format_reward/mean", "format_reward"),
    ("rewards/correctness_reward/mean", "correctness_reward"),
    ("rewards/think_reward/mean", "think_reward"),
    ("kl", "kl"),
    ("entropy", "entropy"),
    ("clip_ratio/region_mean", "clip_ratio"),
]

if not grpo_logs:
    print("No GRPO metric rows were found in trainer.state.log_history.")
else:
    available_columns = [
        (key, label)
        for key, label in metric_columns
        if any(key in row for row in grpo_logs)
    ]

    header = ["step"] + [label for _, label in available_columns]
    widths = {name: max(len(name), 12) for name in header}

    def format_value(value: float | None) -> str:
        if value is None:
            return "-"
        if isinstance(value, int):
            return str(value)
        return f"{value:.4f}"

    print("GRPO metrics by logging step:")
    print("  " + "  ".join(name.ljust(widths[name]) for name in header))

    for row in grpo_logs:
        rendered = {"step": format_value(row.get("step"))}
        for key, label in available_columns:
            rendered[label] = format_value(row.get(key))

        print("  " + "  ".join(rendered[name].ljust(widths[name]) for name in header))

    final_row = grpo_logs[-1]
    print("\nFinal GRPO snapshot:")
    for key, label in available_columns:
        print(f"  {label}: {format_value(final_row.get(key))}")


GRPO metrics by logging step:
  step          reward        reward_std    format_reward  correctness_reward  think_reward  entropy       clip_ratio  
  10            1.5672        0.2690        0.3078         0.6094              0.6500        0.1851        0.0000      
  20            2.4703        0.1053        0.4984         0.9750              0.9969        0.0236        0.0000      
  30            2.4891        0.0619        0.4984         0.9938              0.9969        0.0128        0.0000      
  40            2.4672        0.1114        0.4984         0.9719              0.9969        0.0049        0.0000      
  50            2.4484        0.1274        0.4984         0.9531              0.9969        0.0588        0.0000      
  60            2.4969        0.0177        0.5000         0.9969              1.0000        0.0645        0.0000      
  70            2.4828        0.0601        0.4984         0.9875              0.9969        0.0281        0.0000      
  72      

## Filtered Training Completions (First 5 Train Records)

Load GRPO completion logs and display only rows tied to the first five training prompts.

In [17]:
completion_dir = Path(training_args.output_dir) / "completions"
completion_files = sorted(completion_dir.glob("completions_*.parquet"))

if not completion_files:
    print("No completion parquet files found. Ensure training finished with log_completions=True.")
else:
    completion_ds = Dataset.from_parquet([str(path) for path in completion_files])

    filtered_rows = [
        row for row in completion_ds
        if readable_prompt(row.get("prompt", "")) in first_five_train_prompts
    ]

    if not filtered_rows:
        print("No completion rows matched the first five training prompts.")
    else:
        print(f"Matched {len(filtered_rows)} rows for first 5 training prompts across all steps.")
        print("-" * 120)

        for i, row in enumerate(filtered_rows, start=1):
            prompt = readable_prompt(row.get("prompt", ""))
            completion = row.get("completion", "")
            step = row.get("step", "-")
            reward = row.get("reward", "-")
            format_reward_value = row.get("format_reward", row.get("rewards/format_reward", row.get("rewards/format_reward/mean", "-")))
            correctness_reward_value = row.get("correctness_reward", row.get("rewards/correctness_reward", row.get("rewards/correctness_reward/mean", "-")))
            think_reward_value = row.get("think_reward", row.get("rewards/think_reward", row.get("rewards/think_reward/mean", "-")))

            print(f"Row {i} | step={step} | reward={reward} | format={format_reward_value} | correctness={correctness_reward_value} | think={think_reward_value}")
            print(f"Prompt: {prompt}")
            print(f"Completion: {completion}")
            print("-" * 120)

Generating train split: 256 examples [00:00, 17107.61 examples/s]

No completion rows matched the first five training prompts.


## Post-Training Evaluation

Evaluate the trained model on the same held-out dataset to compare behavior after GRPO and LoRA fine-tuning.

In [18]:
# grab the trained model
trained_model = trainer.model

fine_tuned_eval_result = evaluate_model(
    trained_model,
    tokenizer,
    test_dataset,
    label="After GRPO + LoRA"
)

with start_child_run("fine_tuned_eval"):
    log_eval_result("fine_tuned_eval", fine_tuned_eval_result)

log_metrics(fine_tuned_eval_result["metrics"], prefix="fine_tuned")
parent_run_manager.__exit__(None, None, None)



=== After GRPO + LoRA ===
Answer accuracy:   50/50 = 100.00%
Format compliance: 50/50 = 100.00%
Think compliance:  50/50 = 100.00%
Average reward:    2.500

Sample generations:
------------------------------------------------------------
Prompt:    What is 16 + 3? Respond exactly as <think>16 + 3</think><answer>...</answer>
Expected:  19
Generated: <think>16 + 3</think>
<answer>19</answer>
Predicted: 19
Think reward: 1.0
Reward:    2.5
------------------------------------------------------------
Prompt:    What is 3 + 7? Respond exactly as <think>3 + 7</think><answer>...</answer>
Expected:  10
Generated: <think>3 + 7</think>
<answer>10</answer>
Predicted: 10
Think reward: 1.0
Reward:    2.5
------------------------------------------------------------
Prompt:    What is 12 + 6? Respond exactly as <think>12 + 6</think><answer>...</answer>
Expected:  18
Generated: <think>12 + 6</think>
<answer>18</answer>
Predicted: 18
Think reward: 1.0
Reward:    2.5
------------------------------------

False